DMP PDF
  ↓
1. pdfplumber extraction
  - extract line-level text
  - keep page number, font size, bold, position

  ↓
2. rule-based structure detection
  - Element 1, Element 2 → section
  - A., B., C. → question
  - remaining lines → answer/content

  ↓
3. build narrative JSON
  - load your full RDA + DMPTool extension skeleton
  - keep metadata fields as null
  - fill only narrative.template.section

  ↓
4. save final JSON
  - output looks like your skeleton
  - narrative part contains extracted DMP sections/questions/answers

PDF line label              JSON location
--------------------------------------------------
section                     narrative.template.section[].title

subsection                  narrative.template.section[].question[].text

content after subsection    question[].answer.json.answer[].text

In [9]:
from pathlib import Path
import json
import pandas as pd

from dmpbridge.pdf.pdfplumber_extractor import save_pdfplumber_outputs
from dmpbridge.processing.structure_detector import detect_structure
from dmpbridge.processing.structure_json_builder import save_narrative_json

In [10]:
project_root = Path.cwd().parent

pdf_path = project_root / "data" / "raw_pdfs" / "nih_pre_2026" / "sample.pdf"
skeleton_path = project_root / "schemas" / "rda_dmp_dmptool_extension_skeleton.json"

pdfplumber_json_path = project_root / "data" / "pdfplumber_blocks" / f"{pdf_path.stem}.json"
csv_output_path = project_root / "outputs" / "debug" / f"{pdf_path.stem}_structured_lines.csv"
final_json_path = project_root / "data" / "structure_json" / f"{pdf_path.stem}_narrative.json"

print("PDF exists:", pdf_path.exists())
print("Skeleton exists:", skeleton_path.exists())

PDF exists: True
Skeleton exists: True


In [11]:
blocks = save_pdfplumber_outputs(pdf_path)

print("Extracted lines:", len(blocks))
print("Saved pdfplumber JSON:", pdfplumber_json_path.exists())

[2026-05-04 10:48:49] Extracting line-level text with pdfplumber: sample.pdf
[2026-05-04 10:48:49] Saved line-level JSON: C:\Users\Nahid\dmpbridge\data\pdfplumber_blocks\sample.json
[2026-05-04 10:48:49] Saved extracted text: C:\Users\Nahid\dmpbridge\data\extracted_text\sample.txt
Extracted lines: 141
Saved pdfplumber JSON: True


In [12]:
structured_blocks = detect_structure(blocks)

df = pd.DataFrame(structured_blocks)

df[["page", "line_order", "text", "avg_font_size", "is_bold", "label"]].head(50)

,page,line_order,text,avg_font_size,is_bold,label
0,1,1,DATA MANAGEMENT AND SHARING PLAN,11.04,True,section
1,1,2,An example from an application proposing to co...,11.04,False,content
2,1,3,If any of the proposed research in the applica...,9.00,False,content
3,1,4,for Data Management and Sharing and requires s...,9.00,False,content
4,1,5,application will generate large-scale genomic ...,9.00,False,content
5,1,6,Refer to the detailed instructions in the appl...,9.00,False,content
6,1,7,The Plan is recommended not to exceed two page...,9.00,False,content
7,1,8,There is no “form page” for the Data Managemen...,9.00,False,content
8,1,9,Element 1: Data Type,11.04,True,section
9,1,10,A. Types and amount of scientific data expecte...,11.04,True,subsection


In [13]:
csv_output_path.parent.mkdir(parents=True, exist_ok=True)

df[["page", "line_order", "text", "avg_font_size", "is_bold", "label"]].to_csv(
    csv_output_path,
    index=False,
    encoding="utf-8"
)

print("Saved CSV:", csv_output_path)

Saved CSV: c:\Users\Nahid\dmpbridge\outputs\debug\sample_structured_lines.csv


In [14]:
final_json = save_narrative_json(
    structured_blocks=structured_blocks,
    output_path=final_json_path,
    skeleton_path=skeleton_path
)

print("Saved final JSON:", final_json_path)
print("Number of sections:", len(final_json["narrative"]["template"]["section"]))

[2026-05-04 10:49:13] Saved narrative JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample_narrative.json
Saved final JSON: c:\Users\Nahid\dmpbridge\data\structure_json\sample_narrative.json
Number of sections: 8


In [15]:
for section in final_json["narrative"]["template"]["section"]:
    print(section["order"], section["title"], "| questions:", len(section["question"]))

1 DATA MANAGEMENT AND SHARING PLAN | questions: 0
2 Element 1: Data Type | questions: 3
3 Element 2: Related Tools, Software and/or Code: | questions: 0
4 Element 3: Standards: | questions: 0
5 Element 4: Data Preservation, Access, and Associated Timelines | questions: 3
6 Element 5: Access, Distribution, or Reuse Considerations | questions: 3
7 Element 6: Oversight of Data Management and Sharing: | questions: 0
8 Validation Schedule (this section is required by NIMH) | questions: 0


In [16]:
print(final_json_path.exists())
print(final_json_path)

True
c:\Users\Nahid\dmpbridge\data\structure_json\sample_narrative.json
